<a href="https://colab.research.google.com/github/shashankshekhar28/Rag-based-pdf-QnA-system/blob/main/RAG_Based_PDF_QnA_System_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q --upgrade transformers
!pip install -q pypdf
!pip install -q accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 23.6 MB/s eta 0:00:00


In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [ ]:
from google.colab import files

uploaded = files.upload()

pdf_file = list(uploaded.keys())[0]

print("Uploaded:", pdf_file)

Saving shashank resume.pdf to shashank resume.pdf
Uploaded: shashank resume.pdf


In [ ]:
from pypdf import PdfReader

reader = PdfReader(pdf_file)

text = ""

for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        text += page_text

print("Total characters:", len(text))

Total characters: 1793


In [ ]:
def chunk_text(text, chunk_size=500, overlap=100):

    words = text.split()

    chunks = []
    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = " ".join(words[start:end])

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunks = chunk_text(text)

print("Total chunks:", len(chunks))

Total chunks: 1


In [ ]:
def clean_chunks(chunks):
    return list(set(chunks))


chunks = clean_chunks(chunks)
print("Unique chunks:", len(chunks))

Unique chunks: 1


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5",
    device=device
)

embeddings = embedding_model.encode(
    chunks,
    show_progress_bar=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("Vectors stored:", index.ntotal)

Vectors stored: 1


In [ ]:
def retrieve(query, k=10):

    query_embedding = embedding_model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding),
        k
    )

    results = [chunks[i] for i in indices[0]]

    return results

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device=device
)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [ ]:
def rerank(query, passages, top_k=3):

    pairs = [[query, p] for p in passages]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(passages, scores),
        key=lambda x: x[1],
        reverse=True
    )

    best_passages = [r[0] for r in ranked[:top_k]]

    return best_passages

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="mistralai/Mistral-7B-Instruct-v0.2",
    device=0 if device == "cuda" else -1
)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [ ]:
def build_prompt(question, context):

    prompt = f"""
Answer the question using ONLY the context below.

Context:
{context}

Question: {question}

Answer in one clear sentence.
If the answer is not in the context, say: Not found in document.
"""
    return prompt

In [ ]:
def answer_question(question):

    retrieved = retrieve(question)

    retrieved = clean_chunks(retrieved)

    best_chunks = rerank(question, retrieved)

    context = "\n".join(best_chunks[:2])  # limit context

    prompt = build_prompt(question, context)

    result = generator(
        prompt,
        max_new_tokens=100,
        do_sample=False
    )

    answer = result[0]["generated_text"]

    return answer, context

In [ ]:
question = "summary of this document"

answer, context = answer_question(question)

print("\nANSWER:\n")
print(answer)

print("\nUSED CONTEXT:\n")
print(context)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ANSWER:


Answer the question using ONLY the context below.

Context:
SHASHANK SHEKHAR /githubshashankshekhar28 | /linkedinShashank Shekhar | /envel⌢pess4782129@gmail.com | ♂¶obile+919315208768 Education Manipal University Jaipur Aug 2023 – Present B.Tech in Computer and Communication Engineering ◦ Coursework: Operating Systems , Data Structures, Analysis of Algorithms, Artificial Intel- ligence, Machine Learning, Databases, Web Programming Technologies Languages Python, SQL, C++,HTML,CSS,JavaScript Core CS Data Structures and Algorithms (C,C++,Python), Object-Oriented Programming (C++ and Python), RDBMS (SQL), Software Engineering Principles Tools Git, GitHub, VS Code, PyCharm,Jupyter Notebook,Google Collab Platforms Linux, Windows Soft Skills Leadership, Event Management, Writing, Public Speaking, Time Management Projects Movie Recommendation System GitHub Repository • Built a content-based movie recommendation system using machine learning techniques to suggest similar movies. • Im